# Exploring what I already have

In [ ]:
!pip -q install evaluate rouge_score

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done


In [ ]:
import os
import gc
import random
import evaluate
import numpy as np
import pandas as pd
import torch

from datasets import Dataset
from sklearn.model_selection import train_test_split

from transformers import (
    AutoTokenizer,
    AutoModelForSeq2SeqLM,
    Seq2SeqTrainingArguments,
    Seq2SeqTrainer,
    DataCollatorForSeq2Seq,
    EarlyStoppingCallback
)


In [ ]:
import torch
import psutil
import os

# 1. Check device
if torch.cuda.is_available():
    device = torch.cuda.get_device_name(0)
    print("GPU Device:", device)
    print("GPU Available:", True)

    # GPU memory
    total_gpu = torch.cuda.get_device_properties(0).total_memory / (1024**3)
    allocated_gpu = torch.cuda.memory_allocated(0) / (1024**3)
    reserved_gpu = torch.cuda.memory_reserved(0) / (1024**3)

    print(f"\nTotal GPU Memory: {total_gpu:.2f} GB")
    print(f"GPU Memory Allocated by PyTorch: {allocated_gpu:.2f} GB")
    print(f"GPU Memory Reserved by PyTorch: {reserved_gpu:.2f} GB")
else:
    print("GPU not available. Using CPU.")
    print("CPU:", os.cpu_count(), "logical cores")

# 2. Check system RAM
ram = psutil.virtual_memory()

print("\n--- System RAM ---")
print(f"Total RAM: {ram.total / (1024**3):.2f} GB")
print(f"Used RAM: {ram.used / (1024**3):.2f} GB")
print(f"Available RAM: {ram.available / (1024**3):.2f} GB")
print(f"RAM Usage: {ram.percent:.1f}%")

# 3. Check RAM usage level
if ram.percent >= 90:
    print("WARNING: Very high RAM usage!")
elif ram.percent >= 75:
    print("Notice: RAM usage is high.")
else:
    print("RAM usage is currently within a reasonable range.")

GPU Device: Tesla T4
GPU Available: True

Total GPU Memory: 14.56 GB
GPU Memory Allocated by PyTorch: 2.54 GB
GPU Memory Reserved by PyTorch: 2.73 GB

--- System RAM ---
Total RAM: 50.99 GB
Used RAM: 6.69 GB
Available RAM: 43.68 GB
RAM Usage: 14.3%
RAM usage is currently within a reasonable range.


In [ ]:
%cd "/content/drive/MyDrive/PhD_Documents/5-PhD-July-2026-IMP-Info/2-IITG-Classes/2-NLP-with-LLM/MultiClinSum"
%ls

/content/drive/MyDrive/PhD_Documents/5-PhD-July-2026-IMP-Info/2-IITG-Classes/2-NLP-with-LLM/MultiClinSum
0-Reference-Papers/
1-BART-checkpoints/
1-Extractive-Summarizer-on-592-gs-train-en.ipynb
2-Extractive-Sumamrizer-on-test-en
3-fine-tune-BART.ipynb
multiclinsum_gs_train_en/
multiclinsum_large-scale_train_en/
multiclinsum_test_en/
sentence_segmented_gs_train_en.csv
sentence_segmented_test_en.csv


# Load the dataset (preprocessed data in this case, from the previous steps)

In [ ]:
df = pd.read_csv("sentence_segmented_gs_train_en.csv")

In [ ]:
df.head(2)

,filename,report,summary,cleaned_report,cleaned_summary,r_sentences,su_sentences,r_sent_count,su_sent_count
0,multiclinsum_gs_en_1.txt,A 52-year-old female patient from Wolkiet (Nor...,A 52-year-old known female patient with a toxi...,a 52 year old female patient from wolkiet (nor...,a 52 year old known female patient with a toxi...,['a 52 year old female patient from wolkiet (n...,"[""a 52 year old known female patient with a to...",37,5
1,multiclinsum_gs_en_10.txt,A 61-year-old patient with steroid therapy for...,This case report is about a 61-year-old patien...,a 61 year old patient with steroid therapy for...,this case report is about a 61 year old patien...,['a 61 year old patient with steroid therapy f...,['this case report is about a 61 year old pati...,28,2


In [ ]:
print(df.shape)

(592, 9)


# Load pretrained model

In [ ]:
MODEL_NAME = "facebook/bart-large"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

print("Device:", device)
print("Model loaded:", MODEL_NAME)

Device: cuda
Model loaded: facebook/bart-large


# Token Length Analysis

In [ ]:
report_lengths = [
    len(tokenizer.encode(
        text,
        add_special_tokens=True,
        truncation=False
    ))
    for text in df["cleaned_report"]
]

summary_lengths = [
    len(tokenizer.encode(
        text,
        add_special_tokens=True,
        truncation=False
    ))
    for text in df["cleaned_summary"]
]

print("\nREPORT TOKEN LENGTHS")
print("-" *20)

print(pd.Series(report_lengths).describe(
    percentiles=[0.50, 0.75, 0.90, 0.95, 0.99]
))

print("\nSUMMARY TOKEN LENGTHS")
print("-" *21)

print(pd.Series(summary_lengths).describe(
    percentiles=[0.50, 0.75, 0.90, 0.95, 0.99]
))

print("\nReports exceeding 1024 tokens:",
      sum(n > 1024 for n in report_lengths))
print("Summaries exceeding 512 tokens:",
      sum(n > 512 for n in summary_lengths))

[transformers] Token indices sequence length is longer than the specified maximum sequence length for this model (1111 > 1024). Running this sequence through the model will result in indexing errors



REPORT TOKEN LENGTHS
--------------------
count     592.000000
mean      850.633446
std       582.973708
min       145.000000
50%       751.500000
75%      1025.000000
90%      1343.500000
95%      1468.900000
99%      2312.170000
max      7372.000000
dtype: float64

SUMMARY TOKEN LENGTHS
---------------------
count    592.000000
mean     155.699324
std       73.825083
min       24.000000
50%      142.500000
75%      187.000000
90%      244.000000
95%      269.000000
99%      376.430000
max      990.000000
dtype: float64

Reports exceeding 1024 tokens: 148
Summaries exceeding 512 tokens: 1


# Split the dataset

In [ ]:
train_df, val_df = train_test_split(
    df,
    test_size=50,
    random_state=42,
    shuffle=True
)

train_df = train_df.reset_index(drop=True)
val_df = val_df.reset_index(drop=True)

print("Training examples:", len(train_df))
print("Validation examples:", len(val_df))

assert len(train_df) == 542
assert len(val_df) == 50

Training examples: 542
Validation examples: 50


# Tokenization

In [ ]:
def tokenize_batch(batch):
    inputs = tokenizer(
        batch["cleaned_report"],
        max_length=1024,
        truncation=True
    )

    targets = tokenizer(
        text_target=batch["cleaned_summary"],
        max_length=512,
        truncation=True
    )

    inputs["labels"] = targets["input_ids"]
    return inputs

# convert pandas dataframe to HF Dataset obj
train_ds = Dataset.from_pandas(
    train_df[["cleaned_report", "cleaned_summary"]],
    preserve_index=False
)

val_ds = Dataset.from_pandas(
    val_df[["cleaned_report", "cleaned_summary"]],
    preserve_index=False
)

train_tokenized = train_ds.map(
    tokenize_batch,
    batched=True,
    remove_columns=train_ds.column_names
)

val_tokenized = val_ds.map(
    tokenize_batch,
    batched=True,
    remove_columns=val_ds.column_names
)

print("Training:", len(train_tokenized))
print("Validation:", len(val_tokenized))

Map:   0%|          | 0/542 [00:00<?, ? examples/s]

Map:   0%|          | 0/50 [00:00<?, ? examples/s]

Training: 542
Validation: 50


In [ ]:
# Load BART with its conditional-generation head.
model = AutoModelForSeq2SeqLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.float32
).to(device)

model.config.use_cache = False
model.gradient_checkpointing_enable()
model.enable_input_require_grads()

print("Device:", device)
print("Model class:", type(model).__name__)
print("Architecture:", model.config.architectures)
print("Has language-modeling head:", hasattr(model, "lm_head"))
print("Parameter dtype:", next(model.parameters()).dtype)
print("Parameter device:", next(model.parameters()).device)

data_collator = DataCollatorForSeq2Seq(
    tokenizer=tokenizer,
    model=model,
    padding=True,
    pad_to_multiple_of=8
)

print(
    "Total parameters:",
    f"{sum(p.numel() for p in model.parameters()):,}"
)

print(
    "Trainable parameters:",
    f"{sum(p.numel() for p in model.parameters() if p.requires_grad):,}"
)

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/513 [00:00<?, ?it/s]

Device: cuda
Model class: BartForConditionalGeneration
Architecture: ['BartModel']
Has language-modeling head: True
Parameter dtype: torch.float32
Parameter device: cuda:0
Total parameters: 406,291,456
Trainable parameters: 406,291,456


In [ ]:
# Evaluation metrics

rouge = evaluate.load("rouge")

def compute_metrics(eval_pred):
    predictions, labels = eval_pred

    if isinstance(predictions, tuple):
        predictions = predictions[0]

    predictions = np.where(
        predictions != -100,
        predictions,
        tokenizer.pad_token_id
    )

    labels = np.where(
        labels != -100,
        labels,
        tokenizer.pad_token_id
    )

    decoded_preds = tokenizer.batch_decode(
        predictions,
        skip_special_tokens=True
    )

    decoded_labels = tokenizer.batch_decode(
        labels,
        skip_special_tokens=True
    )

    scores = rouge.compute(
        predictions=[p.strip() for p in decoded_preds],
        references=[s.strip() for s in decoded_labels],
        use_stemmer=True,
        rouge_types=[
            "rouge1",
            "rouge2",
            "rougeL",
            "rougeLsum"
        ]
    )

    return {
        name: round(score * 100, 4)
        for name, score in scores.items()
    }

# Training configuration

In [ ]:
OUTPUT_DIR = (
    "/content/drive/MyDrive/PhD_Documents/5-PhD-July-2026-IMP-Info/2-IITG-Classes/2-NLP-with-LLM/MultiClinSum/1-BART-checkpoints"
    "bart_large_gold592_checkpoints"
)

os.makedirs(OUTPUT_DIR, exist_ok=True)

training_args = Seq2SeqTrainingArguments(
    output_dir=OUTPUT_DIR,

    num_train_epochs=5,
    learning_rate=1e-5,
    weight_decay=0.01,
    warmup_steps=32, # ~6% of train set
    lr_scheduler_type="linear",

    per_device_train_batch_size=1,
    per_device_eval_batch_size=1,
    gradient_accumulation_steps=8,

    fp16=True,
    bf16=False,
    gradient_checkpointing=True,
    max_grad_norm=1.0,

    eval_strategy="epoch",
    save_strategy="epoch",
    logging_strategy="steps",
    logging_steps=20,

    predict_with_generate=True,
    generation_max_length=512,
    generation_num_beams=4,

    metric_for_best_model="rougeL",
    greater_is_better=True,
    load_best_model_at_end=True,
    save_total_limit=2,

    data_seed=42,
    report_to="none",
    optim="adamw_torch"
)

print("Training configuration ready.")


Training configuration ready.


# Fine tune the model

In [ ]:
trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=train_tokenized,
    eval_dataset=val_tokenized,
    processing_class=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
    callbacks=[
        EarlyStoppingCallback(
            early_stopping_patience=2,
            early_stopping_threshold=0.0
        )
    ]
)

gc.collect()
torch.cuda.empty_cache()

train_result = trainer.train()

print("\nTraining metrics:")
print(train_result.metrics)

print("\nBest checkpoint:", trainer.state.best_model_checkpoint)
print("Best validation metric:", trainer.state.best_metric)

Epoch,Training Loss,Validation Loss,Rouge1,Rouge2,Rougel,Rougelsum
1,19.723495,2.201156,39.195200,17.000800,26.617700,26.583400
2,18.241174,2.155022,39.209600,16.981100,26.529000,26.581000
3,17.100342,2.136168,39.376600,17.849200,27.105700,26.977500
4,16.629265,2.120830,40.150800,18.233400,27.592600,27.595100
5,16.350386,2.114504,39.798300,18.098200,27.241000,27.093200


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

[transformers] There were missing keys in the checkpoint model loaded: ['model.encoder.embed_tokens.weight', 'model.decoder.embed_tokens.weight', 'lm_head.weight'].



Training metrics:
{'train_runtime': 2241.3774, 'train_samples_per_second': 1.209, 'train_steps_per_second': 0.152, 'total_flos': 4274952437760000.0, 'train_loss': 17.92320664349724, 'epoch': 5.0}

Best checkpoint: /content/drive/MyDrive/PhD_Documents/5-PhD-July-2026-IMP-Info/2-IITG-Classes/2-NLP-with-LLM/MultiClinSum/1-BART-checkpointsbart_large_gold592_checkpoints/checkpoint-272
Best validation metric: 27.5926


In [ ]:
BEST_MODEL_DIR = (
    "/content/drive/MyDrive/PhD_Documents/5-PhD-July-2026-IMP-Info/2-IITG-Classes/2-NLP-with-LLM/MultiClinSum/1-BART-checkpoints/"
    "bart_large_gold592_best"
)

trainer.save_model(BEST_MODEL_DIR)
tokenizer.save_pretrained(BEST_MODEL_DIR)

print("Best model saved at:", BEST_MODEL_DIR)

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Best model saved at: /content/drive/MyDrive/PhD_Documents/5-PhD-July-2026-IMP-Info/2-IITG-Classes/2-NLP-with-LLM/MultiClinSum/1-BART-checkpoints/bart_large_gold592_best


In [ ]:
%ls

0-Reference-Papers/
1-BART-checkpoints/
1-BART-checkpointsbart_large_gold592_best/
1-BART-checkpointsbart_large_gold592_checkpoints/
1-Extractive-Summarizer-on-592-gs-train-en.ipynb
2-Extractive-Sumamrizer-on-test-en
3-fine-tune-BART.ipynb
multiclinsum_gs_train_en/
multiclinsum_large-scale_train_en/
multiclinsum_test_en/
sentence_segmented_gs_train_en.csv
sentence_segmented_test_en.csv


In [ ]:
!pwd

/content/drive/MyDrive/PhD_Documents/5-PhD-July-2026-IMP-Info/2-IITG-Classes/2-NLP-with-LLM/MultiClinSum


In [ ]:
test_df = pd.read_csv("sentence_segmented_test_en.csv")

In [ ]:
test_df.head(2)

,filename,report,summary,cleaned_report,cleaned_summary,r_sentences,su_sentences,r_sent_count,su_sent_count
0,multiclinsum_test_1000_en.txt,"On July 10, 2022, a 40-year-old woman was admi...",We report the case of a 40-year-old woman with...,"on july 10, 2022, a 40 year old woman was admi...",we report the case of a 40 year old woman with...,"['on july 10, 2022, a 40 year old woman was ad...",['we report the case of a 40 year old woman wi...,39,5
1,multiclinsum_test_1001_en.txt,A 38-year-old woman developed numbness in the ...,We found a 38-year-old female patient with KFS...,a 38 year old woman developed numbness in the ...,we found a 38 year old female patient with kfs...,['a 38 year old woman developed numbness in th...,['we found a 38 year old female patient with k...,33,8


In [ ]:
test_df.shape

(3396, 9)

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

MODEL_DIR = (
    "/content/drive/MyDrive/PhD_Documents/5-PhD-July-2026-IMP-Info/2-IITG-Classes/2-NLP-with-LLM/MultiClinSum/1-BART-checkpoints/"
    "bart_large_gold592_best"
)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

tokenizer = AutoTokenizer.from_pretrained(MODEL_DIR)

model = AutoModelForSeq2SeqLM.from_pretrained(
    MODEL_DIR,
    torch_dtype=torch.float16 if device.type == "cuda" else torch.float32
).to(device)

model.eval()
model.config.use_cache = True

print("Model:", type(model).__name__)
print("Device:", device)
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU")

Loading weights:   0%|          | 0/512 [00:00<?, ?it/s]

Model: BartForConditionalGeneration
Device: cuda
GPU: Tesla T4


# Inference

In [ ]:
from tqdm.auto import tqdm

MAX_INPUT_LENGTH = 1024
MAX_NEW_TOKENS = 512
BATCH_SIZE = 1

def generate_summaries(texts, batch_size=BATCH_SIZE, num_beams=4):
    all_summaries = []

    for start in tqdm(range(0, len(texts), batch_size)):
        batch_texts = texts[start:start + batch_size]

        inputs = tokenizer(
            batch_texts,
            max_length=1024,
            truncation=True,
            padding=True,
            return_tensors="pt"
        )

        inputs = {
            key: value.to(device)
            for key, value in inputs.items()
        }

        with torch.inference_mode():
            output_ids = model.generate(
                **inputs,
                max_new_tokens=512,
                num_beams=num_beams,
                do_sample=False,
                no_repeat_ngram_size=3,
                length_penalty=1.0,
                early_stopping=True
            )

        summaries = tokenizer.batch_decode(
            output_ids,
            skip_special_tokens=True
        )

        all_summaries.extend(summaries)

    return all_summaries


test_reports = test_df["cleaned_report"].tolist()
generated_summaries = generate_summaries(test_reports)

assert len(generated_summaries) == len(test_df)

print("Generated summaries:", len(generated_summaries))

  0%|          | 0/3396 [00:00<?, ?it/s]

[transformers] Both `max_new_tokens` (=512) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=512) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=512) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=512) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/

Generated summaries: 3396


In [ ]:
# !pip install -q rouge_score evaluate

In [ ]:
results_df = test_df.copy()

results_df["generated_summary"] = generated_summaries

OUTPUT_CSV = (
    "/content/drive/MyDrive/PhD_Documents/5-PhD-July-2026-IMP-Info/2-IITG-Classes/2-NLP-with-LLM/MultiClinSum/1-BART-checkpoints/"
    "bart_large_generated_summaries_of_test_set.csv"
)

results_df.to_csv(OUTPUT_CSV, index=False)

print("Predictions saved to:", OUTPUT_CSV)
display(
    results_df[
        ["filename", "cleaned_summary", "generated_summary"]
    ].head(5)
)

Predictions saved to: /content/drive/MyDrive/PhD_Documents/5-PhD-July-2026-IMP-Info/2-IITG-Classes/2-NLP-with-LLM/MultiClinSum/1-BART-checkpoints/bart_large_generated_summaries_of_test_set.csv


,filename,cleaned_summary,generated_summary
0,multiclinsum_test_1000_en.txt,we report the case of a 40 year old woman with...,a 40 year old woman was admitted to the first ...
1,multiclinsum_test_1001_en.txt,we found a 38 year old female patient with kfs...,a 38 year old woman developed numbness in the ...
2,multiclinsum_test_1002_en.txt,we present a case of a 41 year old female that...,a 41 year old female presented with a swelling...
3,multiclinsum_test_1003_en.txt,we present a 26 year old man with a known hist...,we present a 26 year old male with a past medi...
4,multiclinsum_test_1004_en.txt,a 55 year old man was admitted with glasgow co...,a 55 year old man was found unconscious on the...


In [ ]:
for i in range(5):
    row = results_df.iloc[i]

    print("=" * 100)
    print("Filename:", row["filename"])

    print("\nREFERENCE SUMMARY:")
    print(row["summary"])

    print("\nGENERATED SUMMARY:")
    print(row["generated_summary"])

Filename: multiclinsum_test_1000_en.txt

REFERENCE SUMMARY:
We report the case of a 40-year-old woman with intermittent fever for 2 months and gradually accompanied by chills, dry cough, arthralgia, and fatigue. The patient was also diagnosed with depression after fever. She received symptomatic treatment at a regional hospital; however, there was no significant symptom relief. She suddenly developed hemoptysis 1 day prior to arrival at our hospital, where we discovered that her liver, spleen, neck, and axillary lymph nodes were enlarged, and there were multiple nodules in both lungs. The patient was eventually diagnosed with brucellosis after the serum agglutination test and received antibiotic therapy, which provided symptom relief.

GENERATED SUMMARY:
a 40 year old woman was admitted to the first affiliated hospital of xi’an jiaotong university due to intermittent fever for 2 months. she had previously undergone examination and symptomatic treatment at a regional hospital. a month a

In [ ]:
!pip install -q bert-score

In [ ]:
import numpy as np
import pandas as pd
from rouge_score import rouge_scorer

# Use the existing test predictions.
predictions = results_df["generated_summary"].fillna("").astype(str).tolist()

# IMPORTANT:
# Use the reference summaries in the SAME text representation
# as the summaries used for model training/evaluation.
references = results_df["cleaned_summary"].fillna("").astype(str).tolist()

assert len(predictions) == len(references)
assert len(predictions) > 0

scorer = rouge_scorer.RougeScorer(
    ["rouge1", "rouge2", "rougeL"],
    use_stemmer=True
)

all_scores = []

for prediction, reference in zip(predictions, references):
    # rouge_score expects reference first, prediction second.
    scores = scorer.score(reference, prediction)

    row = {}

    for metric in ["rouge1", "rouge2", "rougeL"]:
        row[f"{metric}_precision"] = scores[metric].precision
        row[f"{metric}_recall"] = scores[metric].recall
        row[f"{metric}_f1"] = scores[metric].fmeasure

    all_scores.append(row)

rouge_per_example = pd.DataFrame(all_scores)

# Mean of the individual example scores, on a 0–100 scale.
rouge_summary = pd.DataFrame({
    "Metric": [
        "ROUGE-1",
        "ROUGE-2",
        "ROUGE-L",
    ],
    "Precision": [
        rouge_per_example["rouge1_precision"].mean() * 100,
        rouge_per_example["rouge2_precision"].mean() * 100,
        rouge_per_example["rougeL_precision"].mean() * 100
    ],
    "Recall": [
        rouge_per_example["rouge1_recall"].mean() * 100,
        rouge_per_example["rouge2_recall"].mean() * 100,
        rouge_per_example["rougeL_recall"].mean() * 100
    ],
    "F1": [
        rouge_per_example["rouge1_f1"].mean() * 100,
        rouge_per_example["rouge2_f1"].mean() * 100,
        rouge_per_example["rougeL_f1"].mean() * 100
    ]
})

print("ROUGE RESULTS (mean per-example scores, %)")
display(rouge_summary.round(4))

ROUGE RESULTS (mean per-example scores, %)


,Metric,Precision,Recall,F1
0,ROUGE-1,32.9276,55.6722,38.8042
1,ROUGE-2,15.4161,24.9037,17.8585
2,ROUGE-L,21.8546,36.9631,25.7169


In [ ]:
bertscore = evaluate.load("bertscore")

bert_results = bertscore.compute(
    predictions=predictions,
    references=references,
    model_type="roberta-large",
    device="cuda" if torch.cuda.is_available() else "cpu",
    batch_size=4,
    rescale_with_baseline=False
)

bert_summary = pd.DataFrame({
    "Metric": ["BERTScore"],
    "Precision": [np.mean(bert_results["precision"]) * 100],
    "Recall": [np.mean(bert_results["recall"]) * 100],
    "F1": [np.mean(bert_results["f1"]) * 100]
})

print("BERTSCORE RESULTS (mean per-example scores, %)")
display(bert_summary.round(4))

Loading weights:   0%|          | 0/389 [00:00<?, ?it/s]

[transformers] RobertaModel LOAD REPORT from: roberta-large
Key                       | Status     | 
--------------------------+------------+-
lm_head.bias              | UNEXPECTED | 
lm_head.dense.bias        | UNEXPECTED | 
lm_head.dense.weight      | UNEXPECTED | 
lm_head.layer_norm.bias   | UNEXPECTED | 
lm_head.layer_norm.weight | UNEXPECTED | 
pooler.dense.weight       | MISSING    | 
pooler.dense.bias         | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


BERTSCORE RESULTS (mean per-example scores, %)


,Metric,Precision,Recall,F1
0,BERTScore,83.8537,87.5251,85.622
